In [10]:
# 匯入模組
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.manifold import TSNE

In [11]:
data = pd.read_csv('BackpackPrediction_train.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [13]:
# 刪除類別變項中不含資料的 id
categorical_features = ['Brand', 'Material', 'Size', 'Laptop Compartment', 'Waterproof', 'Style', 'Color']
missing_categorical = data[categorical_features].isnull().any(axis=1)
data = data[~missing_categorical]
data.info() # 再檢查一次是否有缺失值
data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 246686 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    246686 non-null  int64  
 1   Brand                 246686 non-null  object 
 2   Material              246686 non-null  object 
 3   Size                  246686 non-null  object 
 4   Compartments          246686 non-null  float64
 5   Laptop Compartment    246686 non-null  object 
 6   Waterproof            246686 non-null  object 
 7   Style                 246686 non-null  object 
 8   Color                 246686 non-null  object 
 9   Weight Capacity (kg)  246686 non-null  float64
 10  Price                 246686 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 22.6+ MB


,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312


In [16]:
data = data.drop(columns=['id'])

In [17]:
# one-hot coding 以及將布林值轉為0/1
data_encoded = pd.get_dummies(data, drop_first=True)
bool_cols = data_encoded.select_dtypes(include=['bool']).columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

In [18]:
data_encoded = data_encoded.sample(n=50000, random_state=42) # 電腦設備沒有很好，隨機抽取5萬筆
data_encoded = data_encoded.reset_index(drop=True) # 重新設定索引(觀感問題)
data_encoded

,Compartments,Weight Capacity (kg),Price,Brand_Jansport,Brand_Nike,Brand_Puma,Brand_Under Armour,Material_Leather,Material_Nylon,Material_Polyester,...,Size_Small,Laptop Compartment_Yes,Waterproof_Yes,Style_Messenger,Style_Tote,Color_Blue,Color_Gray,Color_Green,Color_Pink,Color_Red
0,3.0,20.034746,25.82827,0,0,0,1,1,0,0,...,1,0,0,0,0,0,0,1,0,0
1,2.0,8.743295,87.11989,1,0,0,0,0,0,1,...,1,1,0,0,0,0,0,0,0,0
2,8.0,28.593976,38.09312,1,0,0,0,0,0,0,...,1,0,0,0,1,0,0,0,0,1
3,2.0,15.188859,127.02658,0,0,0,1,0,1,0,...,0,0,1,1,0,0,1,0,0,0
4,5.0,11.587306,38.48933,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,6.0,16.918217,33.42514,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
49996,5.0,8.637810,39.12619,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
49997,10.0,8.678077,127.31932,0,0,0,1,0,0,0,...,0,0,0,1,0,0,1,0,0,0
49998,6.0,20.589812,123.16063,0,0,0,1,0,0,0,...,1,0,1,0,0,0,0,1,0,0


In [21]:
X_NoPrice = data_encoded.drop(columns='Price')
# 提取Price
y = data_encoded['Price']

In [22]:
# 訓練集、測試集切分
X_train, X_test, y_train, y_test = train_test_split(X_NoPrice, y, test_size=0.2, random_state=42)

# 建立隨機森林模型
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 預測
y_pred = model.predict(X_test)

# 模型評估
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("模型評估結果：")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

模型評估結果：
MSE: 1625.9186126241796
RMSE: 40.32268111899529
R²: -0.0620613294928245


In [24]:
# 讀取測試數據
test_data = pd.read_csv('BackpackPrediction_test.csv')

# 保留 Id 供最終輸出
test_ids = test_data['id'] 
# 移除 Id 欄位（若存在）
test_data.drop(columns=['id'], inplace=True)

# 處理缺失值（填充數值型變數的平均數，類別變數填充 'None'）
for col in test_data.columns:
    if test_data[col].dtype == "object":
        test_data[col].fillna("None", inplace=True)
    else:
        test_data[col].fillna(test_data[col].mean(), inplace=True)

# one-hot coding 以及將布林值轉為0/1
test_data_encoded = pd.get_dummies(test_data, drop_first=True)
bool_cols = test_data_encoded.select_dtypes(include=['bool']).columns
test_data_encoded[bool_cols] = test_data_encoded[bool_cols].astype(int)

# 確保測試數據的欄位與訓練數據匹配（若有缺少的欄位則補0）
missing_cols = set(X_NoPrice.columns) - set(test_data_encoded.columns)
for col in missing_cols:
    test_data_encoded[col] = 0

# 確保欄位順序一致
test_data_encoded = test_data_encoded[X_NoPrice.columns]


In [25]:
test_predictions = model.predict(test_data_encoded)

In [27]:
# 建立提交結果
submission = pd.DataFrame({"id": test_ids, "Price": test_predictions})
submission.set_index("id", inplace=True)

print(submission)

submission.to_csv('BackpackPricePred_RF.csv')

            Price
id               
300000  92.109607
300001  65.701001
300002  80.964265
300003  77.547044
300004  85.805106
...           ...
499995  69.595904
499996  73.293072
499997  99.180322
499998  85.028484
499999  95.277547

[200000 rows x 1 columns]
